# Notebook 2 — Review NLP: Floor & Product Extraction

**Goal:** Extract structured behavioral signals from 70 Yelp reviews to build the foundation for the Agent-Based Model.

**What we extract:**
1. Floor mentions per review → which floors attract attention
2. Product mentions per review → what people actually order
3. Floor co-occurrence → which floors are visited together (proto-transition data)
4. Product-floor associations → what's ordered where
5. Sentiment by floor → satisfaction differences across floors
6. Transition sequences → explicit "we went from X to Y" patterns

**Data:** 70 manually collected Yelp reviews (Apr 2025 – Mar 2026)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
from IPython.display import display

np.random.seed(42)

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    DATA_DIR = Path('/kaggle/input/starbucks-roastery-abm')
    if not DATA_DIR.exists():
        DATA_DIR = Path('/kaggle/input/datasets/shiratoriseto/starbucks-roastery-abm')
else:
    DATA_DIR = Path('../data/raw')

df = pd.read_csv(DATA_DIR / 'reviews_manual.csv')
print(f'Reviews: {len(df)}')
print(f'Date range: {df.date.min()} to {df.date.max()}')
print(f'Avg rating: {df.rating.mean():.2f}')

## Section 1 — Floor Mention Extraction

We use regex patterns to identify which floors each reviewer mentions. Multiple naming conventions exist ("first floor", "1st floor", "ground floor", "main level", etc.).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FLOOR MENTION EXTRACTION
# ══════════════════════════════════════════════════════════════════════

FLOOR_PATTERNS = {
    '1F': r'first\s+floor|1st\s+floor|ground\s+floor|level\s+one|level\s+1\b|main\s+level|bottom\s+floor|main\s+floor|first\s+level',
    '2F': r'second\s+floor|2nd\s+floor|level\s+two|level\s+2\b|bakery|princi|cafe\s+floor',
    '3F': r'third\s+floor|3rd\s+floor|level\s+three|level\s+3\b|experiential|experimental|experience\s+bar|coffee\s+laboratory',
    '4F': r'fourth\s+floor|4th\s+floor|level\s+four|level\s+4\b|cocktail\s+bar|arriviamo|cocktail\s+floor',
    '5F': r'fifth\s+floor|5th\s+floor|rooftop|roof\s+top|terrace|patio',
}

# Extract floor mentions for each review
def extract_floors(text):
    floors = []
    for floor, pattern in FLOOR_PATTERNS.items():
        if re.search(pattern, text, re.IGNORECASE):
            floors.append(floor)
    return floors

df['floors_mentioned'] = df['text'].apply(extract_floors)
df['n_floors_mentioned'] = df['floors_mentioned'].apply(len)

# Floor mention frequency
floor_counts = Counter()
for floors in df['floors_mentioned']:
    for f in floors:
        floor_counts[f] += 1

print('=== Floor Mention Frequency ===')
for floor in ['1F', '2F', '3F', '4F', '5F']:
    count = floor_counts.get(floor, 0)
    pct = count / len(df) * 100
    print(f'  {floor}: {count} reviews ({pct:.0f}%)')

print(f'\nReviews mentioning 0 floors: {(df.n_floors_mentioned == 0).sum()}')
print(f'Reviews mentioning 1 floor:  {(df.n_floors_mentioned == 1).sum()}')
print(f'Reviews mentioning 2+ floors: {(df.n_floors_mentioned >= 2).sum()}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

floors_sorted = ['1F', '2F', '3F', '4F', '5F']
counts = [floor_counts.get(f, 0) for f in floors_sorted]
colors = ['#00704A', '#1E3932', '#D4E9E2', '#CBA258', '#87CEEB']

axes[0].bar(floors_sorted, counts, color=colors, edgecolor='white')
axes[0].set_title('Floor Mention Frequency (n=70 reviews)')
axes[0].set_ylabel('Number of Reviews')
for i, v in enumerate(counts):
    axes[0].text(i, v + 0.5, f'{v} ({v/len(df)*100:.0f}%)', ha='center', fontsize=9)

# Multi-floor distribution
multi_dist = df['n_floors_mentioned'].value_counts().sort_index()
axes[1].bar(multi_dist.index, multi_dist.values, color='#00704A', edgecolor='white')
axes[1].set_title('Number of Floors Mentioned per Review')
axes[1].set_xlabel('Floors Mentioned')
axes[1].set_ylabel('Number of Reviews')

plt.suptitle('Floor Mentions in Yelp Reviews', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 2 — Product Mention Extraction

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PRODUCT MENTION EXTRACTION
# ══════════════════════════════════════════════════════════════════════

PRODUCT_PATTERNS = {
    # Cocktails (4F)
    'Espresso Martini': r'espresso\s+martini',
    'Matcha Margarita': r'matcha\s+margarita',
    'Old Fashioned': r'old\s+fashioned',
    'Cocktail Flight': r'cocktail\s+flight|martini\s+flight|seasonal\s+(cocktail|flight)',
    'Cold Brew Spiced Rum': r'cold\s+brew\s+spiced?\s+rum|spiced?\s+rum',
    'Whiskey Cloud': r'whiskey\s+cloud',
    'Boulevardier': r'boulevardier',
    'Aperol Spritz': r'aperol\s+spritz',
    
    # Coffee (1F/3F)
    'Cold Brew': r'cold\s+brew(?!\s+spiced)(?!\s+malt)',
    'Nitro Cold Brew': r'nitro\s+cold\s+brew',
    'Latte': r'(?<!tiramisu\s)(?<!ube\s)(?<!chai\s)(?<!matcha\s)(?<!caramel\s)latte',
    'Tiramisu Latte': r'tiramisu\s+latte',
    'Ube Coconut Latte': r'ube.{0,10}latte|ube.{0,10}coconut',
    'Salted Caramel Latte': r'salted\s+caramel|caramel\s+bianco',
    'Masala Chai': r'masala\s+chai|chai\s+(latte|tea)',
    'Matcha Latte': r'matcha\s+latte',
    'Americano': r'americano',
    'Espresso Flight': r'espresso\s+flight|coffee\s+flight|origin\s+flight',
    
    # Experience (3F)
    'Siphon': r'siphon',
    'Pour Over': r'pour.over',
    'Chemex': r'chemex',
    'Affogato': r'affogato',
    
    # Food (1F/2F)
    'Croissant/Cornetto': r'croissant|cornetto',
    'Pizza/Focaccia': r'pizza|focaccia|flatbread',
    'Tiramisu (dessert)': r'tiramisu(?!\s+latte|\s+martini|\s+espresso)',
    'Brownie': r'brownie',
    'Cookie': r'cookie',
    'Sandwich': r'sandwich|prosciutto|caprese|porchetta',
    'Olive Oil Cake': r'olive\s+oil\s+cake',
    
    # Retail (1F)
    'Coffee Beans': r'coffee\s+beans|beans|bag\s+of|whiskey\s+barrel.{0,20}(coffee|beans)',
    'Merchandise': r'merch|mug|cup|sweatshirt|ornament|souvenir',
    
    # Whiskey special
    'Whiskey Barrel Cold Brew': r'whiskey\s+barrel.{0,20}cold\s+brew|bourbon.{0,20}(coffee|cold\s+brew)',
}

def extract_products(text):
    products = []
    for product, pattern in PRODUCT_PATTERNS.items():
        if re.search(pattern, text, re.IGNORECASE):
            products.append(product)
    return products

df['products_mentioned'] = df['text'].apply(extract_products)
df['n_products'] = df['products_mentioned'].apply(len)

# Product frequency
product_counts = Counter()
for products in df['products_mentioned']:
    for p in products:
        product_counts[p] += 1

print('=== Top 20 Product Mentions ===')
for product, count in product_counts.most_common(20):
    print(f'  {product:30s}: {count} reviews')

# Visualization
fig, ax = plt.subplots(figsize=(10, 8))
top_products = product_counts.most_common(15)
names = [p[0] for p in top_products]
counts_p = [p[1] for p in top_products]

ax.barh(range(len(names)), counts_p, color='#00704A', edgecolor='white')
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('Number of Reviews')
ax.set_title('Top 15 Product Mentions in Reviews', fontsize=13, fontweight='bold')
ax.invert_yaxis()
for i, v in enumerate(counts_p):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Section 3 — Floor Co-occurrence Matrix

When reviewers mention multiple floors, the co-occurrence tells us which floors are commonly visited together — a proxy for transition patterns.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FLOOR CO-OCCURRENCE MATRIX
# ══════════════════════════════════════════════════════════════════════

floor_labels = ['1F', '2F', '3F', '4F', '5F']
cooccurrence = np.zeros((5, 5), dtype=int)

for floors in df['floors_mentioned']:
    for i, f1 in enumerate(floor_labels):
        for j, f2 in enumerate(floor_labels):
            if f1 in floors and f2 in floors and i != j:
                cooccurrence[i][j] += 1

# Normalize by total multi-floor reviews
multi_floor_reviews = (df['n_floors_mentioned'] >= 2).sum()
cooccurrence_pct = cooccurrence / max(multi_floor_reviews, 1) * 100

print(f'Multi-floor reviews: {multi_floor_reviews}')
print(f'\nCo-occurrence counts:')
cooc_df = pd.DataFrame(cooccurrence, index=floor_labels, columns=floor_labels)
display(cooc_df)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
im1 = axes[0].imshow(cooccurrence, cmap='Greens', vmin=0)
axes[0].set_xticks(range(5)); axes[0].set_yticks(range(5))
axes[0].set_xticklabels(floor_labels); axes[0].set_yticklabels(floor_labels)
axes[0].set_title('Floor Co-occurrence (raw counts)')
axes[0].set_xlabel('Floor')
axes[0].set_ylabel('Floor')
for i in range(5):
    for j in range(5):
        axes[0].text(j, i, str(cooccurrence[i][j]), ha='center', va='center', fontsize=11,
                     color='white' if cooccurrence[i][j] > 10 else 'black')
plt.colorbar(im1, ax=axes[0])

# Percentage
im2 = axes[1].imshow(cooccurrence_pct, cmap='YlOrRd', vmin=0)
axes[1].set_xticks(range(5)); axes[1].set_yticks(range(5))
axes[1].set_xticklabels(floor_labels); axes[1].set_yticklabels(floor_labels)
axes[1].set_title('Floor Co-occurrence (% of multi-floor reviews)')
axes[1].set_xlabel('Floor')
axes[1].set_ylabel('Floor')
for i in range(5):
    for j in range(5):
        axes[1].text(j, i, f'{cooccurrence_pct[i][j]:.0f}%', ha='center', va='center', fontsize=10,
                     color='white' if cooccurrence_pct[i][j] > 30 else 'black')
plt.colorbar(im2, ax=axes[1])

plt.suptitle('Which Floors Are Visited Together?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== Key Co-occurrence Insights ===')
for i in range(5):
    for j in range(i+1, 5):
        if cooccurrence[i][j] >= 5:
            print(f'  {floor_labels[i]} ↔ {floor_labels[j]}: {cooccurrence[i][j]} reviews ({cooccurrence_pct[i][j]:.0f}%)')

## Section 4 — Product-Floor Association

Which products are associated with which floors? This tells us what drives visitors to each floor.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PRODUCT-FLOOR ASSOCIATION
# ══════════════════════════════════════════════════════════════════════

# For each product, count which floors are mentioned in the same review
product_floor_assoc = defaultdict(lambda: Counter())

for _, row in df.iterrows():
    for product in row['products_mentioned']:
        for floor in row['floors_mentioned']:
            product_floor_assoc[product][floor] += 1

# Build association table
assoc_rows = []
for product, floor_counts_p in product_floor_assoc.items():
    total = sum(floor_counts_p.values())
    if total >= 3:  # Only products with 3+ floor-associated mentions
        row = {'product': product, 'total': total}
        for f in floor_labels:
            row[f] = floor_counts_p.get(f, 0)
        # Primary floor
        primary = max(floor_counts_p, key=floor_counts_p.get)
        row['primary_floor'] = primary
        assoc_rows.append(row)

assoc_df = pd.DataFrame(assoc_rows).sort_values('total', ascending=False)
print('=== Product-Floor Associations (3+ mentions) ===')
display(assoc_df[['product', 'primary_floor', 'total', '1F', '2F', '3F', '4F', '5F']])

# Heatmap
if len(assoc_df) > 0:
    heat_data = assoc_df.set_index('product')[floor_labels].head(12)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(heat_data.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(5))
    ax.set_xticklabels(floor_labels)
    ax.set_yticks(range(len(heat_data)))
    ax.set_yticklabels(heat_data.index, fontsize=9)
    ax.set_title('Product-Floor Association Heatmap', fontsize=13, fontweight='bold')
    for i in range(len(heat_data)):
        for j in range(5):
            ax.text(j, i, str(heat_data.values[i][j]), ha='center', va='center', fontsize=9)
    plt.colorbar(im, label='Co-mention count')
    plt.tight_layout()
    plt.show()

## Section 5 — Transition Sequence Extraction

Some reviews describe explicit movement patterns: "we started on the 2nd floor, then went up to the 4th floor." We extract these as directed transitions.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TRANSITION SEQUENCE EXTRACTION
# ══════════════════════════════════════════════════════════════════════

# Find ordered floor mentions in text (by position)
def extract_floor_sequence(text):
    """Extract floors in the order they appear in the text."""
    mentions = []
    for floor, pattern in FLOOR_PATTERNS.items():
        for match in re.finditer(pattern, text, re.IGNORECASE):
            mentions.append((match.start(), floor))
    # Sort by position in text
    mentions.sort(key=lambda x: x[0])
    # Deduplicate consecutive same-floor mentions
    sequence = []
    for _, floor in mentions:
        if not sequence or sequence[-1] != floor:
            sequence.append(floor)
    return sequence

df['floor_sequence'] = df['text'].apply(extract_floor_sequence)
df['has_sequence'] = df['floor_sequence'].apply(lambda x: len(x) >= 2)

# Extract directed transitions
transition_counts = Counter()
for seq in df['floor_sequence']:
    for i in range(len(seq) - 1):
        transition = f'{seq[i]} → {seq[i+1]}'
        transition_counts[transition] += 1

print(f'Reviews with floor sequences (2+ floors in order): {df.has_sequence.sum()}')
print(f'\n=== Observed Transitions ===')
for transition, count in transition_counts.most_common():
    print(f'  {transition}: {count}')

# Show example sequences
print(f'\n=== Example Floor Sequences ===')
for _, row in df[df.has_sequence].head(10).iterrows():
    seq_str = ' → '.join(row['floor_sequence'])
    print(f'  {seq_str}  (rating: {row["rating"]}★)')

# Build transition matrix from observed sequences
transition_matrix = np.zeros((5, 5))
for seq in df['floor_sequence']:
    for i in range(len(seq) - 1):
        from_idx = floor_labels.index(seq[i])
        to_idx = floor_labels.index(seq[i+1])
        transition_matrix[from_idx][to_idx] += 1

# Normalize rows
row_sums = transition_matrix.sum(axis=1, keepdims=True)
transition_matrix_norm = np.divide(transition_matrix, row_sums, where=row_sums > 0, out=np.zeros_like(transition_matrix))

print(f'\n=== Observed Transition Matrix (normalized) ===')
trans_df = pd.DataFrame(transition_matrix_norm.round(3), index=floor_labels, columns=floor_labels)
display(trans_df)

## Section 6 — Sentiment by Floor

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SENTIMENT BY FLOOR (using rating as proxy)
# ══════════════════════════════════════════════════════════════════════

# Simple approach: average rating of reviews that mention each floor
floor_sentiment = {}
for floor in floor_labels:
    mask = df['floors_mentioned'].apply(lambda x: floor in x)
    if mask.sum() > 0:
        floor_sentiment[floor] = {
            'avg_rating': df[mask]['rating'].mean(),
            'n_reviews': mask.sum(),
            'pct_5star': (df[mask]['rating'] == 5).mean() * 100,
        }

sentiment_df = pd.DataFrame(floor_sentiment).T
print('=== Sentiment by Floor ===')
display(sentiment_df.round(2))

# Positive/negative keyword analysis per floor
positive_words = r'amazing|incredible|delicious|love|great|best|perfect|outstanding|fantastic|wonderful|fabulous|phenomenal'
negative_words = r'disappoint|overpriced|crowded|expensive|mediocre|not.{0,10}good|waste|bad|worst|terrible'

for floor in floor_labels:
    mask = df['floors_mentioned'].apply(lambda x: floor in x)
    if mask.sum() > 0:
        texts = ' '.join(df[mask]['text'].values).lower()
        pos = len(re.findall(positive_words, texts))
        neg = len(re.findall(negative_words, texts))
        total = pos + neg
        if total > 0:
            print(f'  {floor}: +{pos} / -{neg} ({pos/total*100:.0f}% positive)')

# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
floors_plot = list(floor_sentiment.keys())
ratings = [floor_sentiment[f]['avg_rating'] for f in floors_plot]
n_reviews = [floor_sentiment[f]['n_reviews'] for f in floors_plot]

bars = ax.bar(floors_plot, ratings, color=colors[:len(floors_plot)], edgecolor='white')
ax.set_ylim(3.5, 5.0)
ax.set_title('Average Rating by Floor Mentioned', fontsize=13, fontweight='bold')
ax.set_ylabel('Average Rating (★)')
ax.axhline(df['rating'].mean(), color='red', linestyle='--', alpha=0.5, label=f'Overall avg: {df["rating"].mean():.2f}★')
ax.legend()
for i, (r, n) in enumerate(zip(ratings, n_reviews)):
    ax.text(i, r + 0.05, f'{r:.2f}★\n(n={n})', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Section 7 — Save Extracted Data for ABM

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SAVE EXTRACTED DATA
# ══════════════════════════════════════════════════════════════════════

OUT_DIR = Path('../data/processed') if not ON_KAGGLE else Path('/kaggle/working')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Enriched reviews
df_save = df.copy()
df_save['floors_mentioned'] = df_save['floors_mentioned'].apply(lambda x: '|'.join(x) if x else '')
df_save['products_mentioned'] = df_save['products_mentioned'].apply(lambda x: '|'.join(x) if x else '')
df_save['floor_sequence'] = df_save['floor_sequence'].apply(lambda x: '|'.join(x) if x else '')
df_save.to_csv(OUT_DIR / 'reviews_enriched.csv', index=False)
print(f'Saved: reviews_enriched.csv ({len(df_save)} rows)')

# 2. Observed transition matrix
trans_save = pd.DataFrame(transition_matrix_norm, index=floor_labels, columns=floor_labels)
trans_save.to_csv(OUT_DIR / 'transition_matrix_observed.csv')
print(f'Saved: transition_matrix_observed.csv')

# 3. Product-floor associations
if len(assoc_df) > 0:
    assoc_df.to_csv(OUT_DIR / 'product_floor_associations.csv', index=False)
    print(f'Saved: product_floor_associations.csv ({len(assoc_df)} products)')

# 4. Floor sentiment summary
sentiment_df.to_csv(OUT_DIR / 'floor_sentiment.csv')
print(f'Saved: floor_sentiment.csv')

print('\n=== Summary ===')
print(f'Reviews processed: {len(df)}')
print(f'Floor mentions extracted: {sum(floor_counts.values())}')
print(f'Product mentions extracted: {sum(product_counts.values())}')
print(f'Directed transitions observed: {sum(transition_counts.values())}')
print(f'Multi-floor reviews: {df.has_sequence.sum()}')

## Limitations

1. **Small sample size (n=70)** — Transition probabilities have wide confidence intervals. Sensitivity analysis in Notebook 6 will quantify this.
2. **Floor confusion in reviews** — Some reviewers misidentify floors (e.g., calling 4F "3rd floor"). We handle this via keyword matching but some noise remains.
3. **Selection bias** — Yelp reviewers skew toward tourists and first-time visitors. Regular locals may have different patterns.
4. **No temporal resolution** — We know which floors were visited but not exact timing or duration (except rare mentions like "stayed 2 hours").
5. **Rating as sentiment proxy** — Star rating is a crude sentiment measure. A dedicated sentiment model would be more nuanced but overkill for 70 reviews.

## Next Steps

- **Notebook 3:** Combine observed transitions with physical adjacency prior to build calibrated transition matrix for ABM
- **Notebook 4:** Agent-Based Model simulation using the calibrated matrix